[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/10.%20The%20Dual-Cycling%20Quay%20Crane%20Problem/P10-Tier-5_executed.ipynb)

---


# 10. The Dual-Cycling Quay Crane Problem

## Tier 5 — Integrated Digital Twin: A System-of-Systems Simulation

### Goal

Build a small, reproducible simulation that represents quay cranes and connected terminal subsystems in a shared digital state. The notebook demonstrates how subsystem synchronization, rule-based global recommendations, and simple horizon-based predictions can be organized in one integrated framework.

### Inputs and outputs

The input is a terminal configuration: counts of cranes, vessels, yard blocks, and trucks; a container total; and nominal simulation and synchronization settings. The code initializes subsystem state, advances that state using seeded random draws, records synchronization and optimization histories, and produces a forecast from the simulated state.

### Scope and modeling assumptions

This is an instructional simulation, not a production digital twin. It does not connect to sensors, external terminal systems, a network, or a physical crane. Network latency, synchronization success, and data consistency are sampled values; they are not measured service-level results. The nominal one-second synchronization interval and 30-second optimization interval organize simulated time, but the notebook does not enforce real-time execution or sleep between steps. The update rules and forecast coefficients are illustrative assumptions, not empirically calibrated or validated models. The yard state represents aggregate yard conditions, and the resource count covers cranes and trucks rather than a measured quantity of yard space.

The optimization component retains the notebook's rule-based sequence: identify bottlenecks and opportunities, generate recommendations, and apply those recommendations to the simulated state. It is not a mathematical programming solver and does not establish a globally optimal operating plan. The prediction component is a deterministic calculation from the current state and a horizon-dependent coefficient; it does not estimate forecast accuracy from historical observations.

### Method at a glance

At each synchronization step, the simulator updates subsystem timestamps, evolves the modeled states, aggregates selected measures, and records synchronization metrics. At the configured optimization interval, it checks thresholds and applies any resulting recommendations. A final forecast estimates cumulative container throughput and truck efficiency over a requested horizon.

The main simulation work at each synchronization step is proportional to the number of stored subsystems, denoted by \(S\). The rule-based optimization also scans subsystem state, so its work is proportional to \(S\) for this implementation. These bounds describe the code's state scans; they do not imply that the model captures all computational or operational costs of a real terminal.

In [ ]:
from dataclasses import dataclass, field
import random
import time
from typing import Any, Dict, List, Tuple

import numpy as np

# Fixed seeds make the illustrative random state updates reproducible.
np.random.seed(42)
random.seed(42)

print("Simulation dependencies are ready.")

### Digital Twin Architecture Definition

A **subsystem state** is the information the simulation stores about one represented part of the terminal at a particular simulated time. Keeping subsystem-specific fields in separate data classes makes the model's interfaces explicit: crane state, vessel state, yard state, truck state, and resource state can each carry different information while sharing common synchronization fields.

In Python, a data class generates an initializer from its declared fields. The base fields below are all required, so subclasses can add required fields without placing a required argument after an inherited optional argument. Each constructor supplies the common health and synchronization values explicitly.

In [ ]:
@dataclass
class SubsystemState:
    """Common timestamped state shared by terminal subsystems."""

    subsystem_id: str
    timestamp: float
    state_data: Dict[str, Any]
    health_status: str
    last_sync: float


@dataclass
class QuayCraneState(SubsystemState):
    """Simulated operating state for one quay crane."""

    crane_id: int
    position: Tuple[float, float]
    load_status: str
    operation_type: str
    operation_progress: float
    dual_cycle_available: bool = False
    interference_zone: Tuple[float, float] = (0.0, 0.0)


@dataclass
class VesselScheduleState(SubsystemState):
    """Simulated schedule and processing state for one vessel."""

    vessel_id: str
    berth_position: int
    total_containers: int
    containers_processed: int
    estimated_departure: float
    stability_metrics: Dict[str, float]
    hatch_config: Dict[int, str]


@dataclass
class YardManagementState(SubsystemState):
    """Aggregate yard conditions represented by the simulation."""

    block_id: str
    utilization: float
    truck_queue_length: int
    available_positions: int
    congestion_level: str
    predicted_congestion: Dict[str, float] = field(default_factory=dict)


@dataclass
class TruckDispatchState(SubsystemState):
    """Simulated fleet activity and dispatch measures."""

    fleet_size: int
    active_trucks: int
    average_wait_time: float
    dispatch_efficiency: float
    route_congestion: Dict[str, float] = field(default_factory=dict)


@dataclass
class ResourceAllocationState(SubsystemState):
    """Aggregate allocation measures for countable resources."""

    total_resources: int
    allocated_resources: int
    utilization_rate: float
    allocation_efficiency: float
    bottleneck_resources: List[str] = field(default_factory=list)


print("Digital twin state data classes are defined.")

### System-of-Systems Integration Framework

The integration class stores the subsystem objects and coordinates the simulation's sequence. Its main operations are deliberately separated:

1. `_initialize_subsystems` creates the initial digital representations.
2. `synchronize_subsystems` advances and aggregates their state.
3. `run_global_optimization` identifies conditions, creates rule-based recommendations, and applies them.
4. `predict_operations` calculates a horizon-based estimate from the current aggregate state.

A **synchronization metric** here is a simulated quantity recorded during a state update. In particular, `avg_latency` is sampled from the stated illustrative latency range, whereas `sync_latency` measures the local wall-clock time taken by the synchronization method. Neither is a measurement of communication with a real terminal.

The bottleneck and opportunity checks use explicit thresholds. For example, if \(A\) cranes are active out of \(C\) configured cranes, the low-utilization rule is triggered when

\[
\frac{A}{C} < 0.7.
\]

The forecast uses a horizon-dependent factor \(a(h)\) and an illustrative base rate of 8 containers per five-minute interval. For horizon \(h\) minutes and current processed total \(P\), the calculation is

\[
\widehat{P}(h) = \min\!\left(P_{\mathrm{total}},\; P + \frac{h}{5}\,8\,a(h)\right),
\]

where \(P_{\mathrm{total}}\) is the configured container total. The forecast efficiency is the current truck efficiency multiplied by the same factor. The interval around predicted throughput is formed using the horizon-specific uncertainty \(u(h)\):

\[
\left[\widehat{P}(h)(1-u(h)),\; \min\!\left(P_{\mathrm{total}},\widehat{P}(h)(1+u(h))\right)\right].
\]

These equations describe the code's illustrative calculations; the factors are assumptions, not claims about measured forecast performance.

In [ ]:
class IntegratedDigitalTwin:
    """Coordinate simulated terminal subsystems and their simple heuristics."""

    def __init__(self, terminal_config: Dict[str, Any]):
        required_keys = {
            'num_cranes',
            'num_vessels',
            'num_yard_blocks',
            'num_trucks',
            'total_containers',
            'simulation_duration',
            'sync_frequency',
            'optimization_frequency',
        }
        missing_keys = required_keys.difference(terminal_config)
        if missing_keys:
            raise ValueError(f"Missing terminal configuration keys: {sorted(missing_keys)}")

        for key in ('num_cranes', 'num_vessels', 'num_yard_blocks', 'num_trucks', 'total_containers'):
            if terminal_config[key] <= 0:
                raise ValueError(f"{key} must be positive.")
        if terminal_config['simulation_duration'] <= 0:
            raise ValueError("simulation_duration must be positive.")
        if terminal_config['sync_frequency'] <= 0:
            raise ValueError("sync_frequency must be positive.")
        if terminal_config['optimization_frequency'] <= 0:
            raise ValueError("optimization_frequency must be positive.")

        self.terminal_config = terminal_config
        self.subsystems: Dict[str, SubsystemState] = {}
        self.global_state: Dict[str, Any] = {}
        self.sync_history: List[Dict[str, Any]] = []
        self.optimization_history: List[Dict[str, Any]] = []
        self.prediction_history: List[Dict[str, Any]] = []

        self.sync_latency = 0.0
        self.uptime_percentage = 100.0
        self.optimization_frequency = float(terminal_config['optimization_frequency'])
        self._initialize_subsystems()

    def _initialize_subsystems(self) -> None:
        """Create initial state objects from the supplied terminal configuration."""
        current_time = time.time()

        for crane_id in range(self.terminal_config['num_cranes']):
            crane_state = QuayCraneState(
                subsystem_id=f'qc_{crane_id}',
                timestamp=current_time,
                state_data={'active': True},
                health_status='operational',
                last_sync=current_time,
                crane_id=crane_id,
                position=(crane_id * 2 + 1, 0.0),
                load_status='empty',
                operation_type='wait',
                operation_progress=0.0,
                interference_zone=(crane_id * 2, crane_id * 2 + 2),
            )
            self.subsystems[crane_state.subsystem_id] = crane_state

        vessel_count = self.terminal_config['num_vessels']
        container_count = self.terminal_config['total_containers']
        containers_per_vessel, remainder = divmod(container_count, vessel_count)
        for vessel_index in range(vessel_count):
            vessel_containers = containers_per_vessel + (vessel_index < remainder)
            vessel_id = f'VSL{vessel_index + 1:03d}'
            vessel_state = VesselScheduleState(
                subsystem_id=f'vessel_{vessel_index}',
                timestamp=current_time,
                state_data={'active_vessels': vessel_count},
                health_status='operational',
                last_sync=current_time,
                vessel_id=vessel_id,
                berth_position=vessel_index + 1,
                total_containers=vessel_containers,
                containers_processed=0,
                estimated_departure=current_time + 7200,
                stability_metrics={'list': 0.1, 'trim': 0.05},
                hatch_config={bay: 'closed' for bay in range(1, 19)},
            )
            self.subsystems[vessel_state.subsystem_id] = vessel_state

        yard_state = YardManagementState(
            subsystem_id='yard_management',
            timestamp=current_time,
            state_data={'total_blocks': self.terminal_config['num_yard_blocks']},
            health_status='operational',
            last_sync=current_time,
            block_id='YB001',
            utilization=0.7,
            truck_queue_length=3,
            available_positions=12,
            congestion_level='medium',
        )
        self.subsystems[yard_state.subsystem_id] = yard_state

        truck_count = self.terminal_config['num_trucks']
        truck_state = TruckDispatchState(
            subsystem_id='truck_dispatch',
            timestamp=current_time,
            state_data={'total_routes': 8},
            health_status='operational',
            last_sync=current_time,
            fleet_size=truck_count,
            active_trucks=int(truck_count * 0.8),
            average_wait_time=5.2,
            dispatch_efficiency=0.85,
        )
        self.subsystems[truck_state.subsystem_id] = truck_state

        countable_resources = self.terminal_config['num_cranes'] + truck_count
        resource_state = ResourceAllocationState(
            subsystem_id='resource_allocation',
            timestamp=current_time,
            state_data={'resource_types': ['cranes', 'trucks']},
            health_status='operational',
            last_sync=current_time,
            total_resources=countable_resources,
            allocated_resources=int(countable_resources * 0.75),
            utilization_rate=0.75,
            allocation_efficiency=0.82,
        )
        self.subsystems[resource_state.subsystem_id] = resource_state

    def synchronize_subsystems(self, current_time: float) -> Dict[str, float]:
        """Advance subsystem state and record illustrative synchronization metrics."""
        sync_start = time.time()
        sync_metrics = {
            'success_rate': 0.0,
            'avg_latency': 0.0,
            'data_consistency': 0.0,
        }

        for subsystem in self.subsystems.values():
            # This draw represents a modeled network delay; no real network call occurs.
            sync_metrics['avg_latency'] += np.random.uniform(0.01, 0.05)
            subsystem.timestamp = current_time
            subsystem.last_sync = current_time
            self._evolve_subsystem_state(subsystem, current_time)

        sync_metrics['avg_latency'] /= len(self.subsystems)
        sync_metrics['success_rate'] = float(np.clip(0.95 + np.random.normal(0, 0.02), 0.0, 1.0))
        sync_metrics['data_consistency'] = float(np.clip(0.97 + np.random.normal(0, 0.01), 0.0, 1.0))

        self._update_global_state(current_time)
        self.sync_history.append({
            'timestamp': current_time,
            'metrics': sync_metrics.copy(),
        })
        self.sync_latency = time.time() - sync_start
        return sync_metrics

    def _evolve_subsystem_state(
        self, subsystem: SubsystemState, current_time: float
    ) -> None:
        """Apply the illustrative state-transition rules for the subsystem type."""
        if isinstance(subsystem, QuayCraneState):
            if subsystem.operation_type != 'wait':
                subsystem.operation_progress = min(
                    1.0,
                    subsystem.operation_progress + np.random.uniform(0.05, 0.15),
                )
                if subsystem.operation_progress >= 1.0:
                    self._assign_next_crane_operation(subsystem)
            elif np.random.random() < 0.3:
                self._assign_next_crane_operation(subsystem)

        elif isinstance(subsystem, VesselScheduleState):
            processed_increment = np.random.poisson(2)
            subsystem.containers_processed = min(
                subsystem.total_containers,
                subsystem.containers_processed + processed_increment,
            )
            subsystem.stability_metrics['list'] += np.random.normal(0, 0.01)
            subsystem.stability_metrics['trim'] += np.random.normal(0, 0.005)

        elif isinstance(subsystem, YardManagementState):
            subsystem.utilization = float(np.clip(
                subsystem.utilization + np.random.normal(0, 0.02), 0.0, 1.0
            ))
            subsystem.truck_queue_length = max(
                0, subsystem.truck_queue_length + np.random.randint(-1, 2)
            )
            if subsystem.truck_queue_length < 3:
                subsystem.congestion_level = 'low'
            elif subsystem.truck_queue_length < 6:
                subsystem.congestion_level = 'medium'
            else:
                subsystem.congestion_level = 'high'

        elif isinstance(subsystem, TruckDispatchState):
            subsystem.active_trucks = int(np.clip(
                subsystem.active_trucks + np.random.randint(-2, 3),
                0,
                subsystem.fleet_size,
            ))
            subsystem.average_wait_time = max(
                1.0, subsystem.average_wait_time + np.random.normal(0, 0.5)
            )
            subsystem.dispatch_efficiency = float(np.clip(
                subsystem.dispatch_efficiency + np.random.normal(0, 0.02),
                0.0,
                1.0,
            ))

    def _assign_next_crane_operation(self, crane: QuayCraneState) -> None:
        """Sample the next crane operation and keep its load label consistent."""
        operations = ['unload', 'load', 'move', 'wait']
        if crane.dual_cycle_available and np.random.random() < 0.7:
            crane.operation_type = str(np.random.choice(['unload', 'load']))
        else:
            crane.operation_type = str(np.random.choice(operations, p=[0.3, 0.3, 0.2, 0.2]))

        if crane.operation_type == 'unload':
            crane.load_status = 'import'
        elif crane.operation_type == 'load':
            crane.load_status = 'export'
        else:
            crane.load_status = 'empty'

        crane.operation_progress = 0.0
        crane.dual_cycle_available = bool(np.random.random() < 0.6)

    def _update_global_state(self, current_time: float) -> None:
        """Aggregate selected subsystem values for system-level checks and forecasts."""
        vessels = [
            state for state in self.subsystems.values()
            if isinstance(state, VesselScheduleState)
        ]
        cranes = [
            state for state in self.subsystems.values()
            if isinstance(state, QuayCraneState)
        ]
        yards = [
            state for state in self.subsystems.values()
            if isinstance(state, YardManagementState)
        ]
        trucks = [
            state for state in self.subsystems.values()
            if isinstance(state, TruckDispatchState)
        ]
        resources = [
            state for state in self.subsystems.values()
            if isinstance(state, ResourceAllocationState)
        ]

        self.global_state = {
            'timestamp': current_time,
            'total_containers_processed': sum(
                vessel.containers_processed for vessel in vessels
            ),
            'total_containers': sum(vessel.total_containers for vessel in vessels),
            'active_cranes': sum(
                crane.operation_type != 'wait' for crane in cranes
            ),
            'yard_utilization': float(np.mean([yard.utilization for yard in yards])),
            'truck_efficiency': float(np.mean(
                [truck.dispatch_efficiency for truck in trucks]
            )),
            'resource_utilization': float(np.mean(
                [resource.utilization_rate for resource in resources]
            )),
        }

    def run_global_optimization(self, current_time: float) -> Dict[str, Any]:
        """Run the rule-based identify, recommend, and apply sequence."""
        optimization_start = time.time()
        bottlenecks = self._identify_bottlenecks()
        opportunities = self._identify_optimization_opportunities()
        recommendations = self._generate_optimization_recommendations(
            bottlenecks, opportunities
        )
        optimization_results = self._apply_optimizations(recommendations)
        results = {
            'timestamp': current_time,
            'optimization_time': time.time() - optimization_start,
            'bottlenecks': bottlenecks,
            'opportunities': opportunities,
            'recommendations': recommendations,
            'results': optimization_results,
        }
        self.optimization_history.append(results)
        return results

    def _identify_bottlenecks(self) -> List[str]:
        """Apply the specified utilization and congestion thresholds."""
        bottlenecks = []
        active_cranes = self.global_state['active_cranes']
        total_cranes = self.terminal_config['num_cranes']
        if active_cranes / total_cranes < 0.7:
            bottlenecks.append('Low crane utilization')

        yard_congestion = any(
            state.congestion_level == 'high'
            for state in self.subsystems.values()
            if isinstance(state, YardManagementState)
        )
        if yard_congestion:
            bottlenecks.append('High yard congestion')

        if self.global_state['truck_efficiency'] < 0.8:
            bottlenecks.append('Low truck dispatch efficiency')
        return bottlenecks

    def _identify_optimization_opportunities(self) -> List[str]:
        """Identify dual-cycling and aggregate resource opportunities."""
        opportunities = []
        dual_cycle_available = sum(
            state.dual_cycle_available
            for state in self.subsystems.values()
            if isinstance(state, QuayCraneState)
        )
        if dual_cycle_available > 0:
            opportunities.append(
                f'Dual-cycling opportunities: {dual_cycle_available} cranes'
            )

        if self.global_state['resource_utilization'] < 0.9:
            opportunities.append('Resource reallocation potential')
        opportunities.append('Predictive scheduling optimization')
        return opportunities

    def _generate_optimization_recommendations(
        self, bottlenecks: List[str], opportunities: List[str]
    ) -> List[Dict[str, Any]]:
        """Map detected conditions to the notebook's fixed recommendations."""
        recommendations = []
        if 'Low crane utilization' in bottlenecks:
            recommendations.append({
                'type': 'crane_reassignment',
                'action': 'Reassign idle cranes to high-priority operations',
                'expected_improvement': 15,
            })
        if 'High yard congestion' in bottlenecks:
            recommendations.append({
                'type': 'yard_optimization',
                'action': 'Pre-position yard equipment and optimize truck routes',
                'expected_improvement': 12,
            })
        if any('dual-cycling' in opportunity for opportunity in opportunities):
            recommendations.append({
                'type': 'dual_cycling',
                'action': 'Increase dual-cycling operations through coordinated scheduling',
                'expected_improvement': 25,
            })
        return recommendations

    def _apply_optimizations(
        self, recommendations: List[Dict[str, Any]]
    ) -> Dict[str, float]:
        """Apply recommendation rules to the simulated subsystem state."""
        results = {
            'throughput_improvement': 0.0,
            'efficiency_gain': 0.0,
            'dual_cycling_increase': 0.0,
        }
        for recommendation in recommendations:
            if recommendation['type'] == 'dual_cycling':
                for state in self.subsystems.values():
                    if isinstance(state, QuayCraneState):
                        state.dual_cycle_available = True
                results['dual_cycling_increase'] = (
                    recommendation['expected_improvement'] / 100
                )

            elif recommendation['type'] == 'crane_reassignment':
                for state in self.subsystems.values():
                    if isinstance(state, QuayCraneState) and state.operation_type == 'wait':
                        state.operation_type = str(np.random.choice(['unload', 'load']))
                        state.load_status = (
                            'import' if state.operation_type == 'unload' else 'export'
                        )
                        state.operation_progress = 0.0
                results['throughput_improvement'] = (
                    recommendation['expected_improvement'] / 100
                )

            elif recommendation['type'] == 'yard_optimization':
                for state in self.subsystems.values():
                    if isinstance(state, YardManagementState):
                        state.truck_queue_length = max(0, state.truck_queue_length - 2)
                        if state.congestion_level == 'high':
                            state.congestion_level = 'medium'
                results['efficiency_gain'] = (
                    recommendation['expected_improvement'] / 100
                )
        return results

    def predict_operations(self, horizon_minutes: int) -> Dict[str, Any]:
        """Calculate an illustrative throughput and efficiency forecast."""
        if horizon_minutes <= 0:
            raise ValueError("horizon_minutes must be positive.")
        if not self.global_state:
            raise RuntimeError("Synchronize the subsystems before requesting a prediction.")

        current_throughput = self.global_state['total_containers_processed']
        total_containers = self.global_state['total_containers']
        current_efficiency = self.global_state['truck_efficiency']

        if horizon_minutes <= 15:
            accuracy_factor = 0.942
            uncertainty = 0.05
        elif horizon_minutes <= 60:
            accuracy_factor = 0.876
            uncertainty = 0.12
        else:
            accuracy_factor = 0.75
            uncertainty = 0.25

        time_steps = horizon_minutes / 5
        predicted_throughput = min(
            total_containers,
            current_throughput + time_steps * 8 * accuracy_factor,
        )
        predicted_efficiency = current_efficiency * accuracy_factor
        ci_lower = max(0.0, predicted_throughput * (1 - uncertainty))
        ci_upper = min(total_containers, predicted_throughput * (1 + uncertainty))

        risk_factors = []
        if self.global_state['yard_utilization'] > 0.85:
            risk_factors.append('High yard utilization may cause delays')
        if self.global_state['active_cranes'] < self.terminal_config['num_cranes'] * 0.7:
            risk_factors.append('Low crane activity affecting throughput')

        prediction_results = {
            'horizon_minutes': horizon_minutes,
            'predicted_throughput': int(predicted_throughput),
            'predicted_efficiency': predicted_efficiency,
            'confidence_interval': (int(ci_lower), int(ci_upper)),
            'risk_factors': risk_factors,
        }
        self.prediction_history.append(prediction_results)
        return prediction_results


print("Integrated digital twin class is defined.")

### Digital Twin Simulation Setup

The configuration below preserves the example's terminal size and nominal time settings. The simulation advances in discrete increments of `sync_frequency`; it does not wait in real time. At each increment it calls the synchronization method, and it invokes the global rule-based optimization whenever the elapsed simulated time reaches the configured optimization interval.

The configured vessel count is represented by one vessel-state object per vessel. The total container count is divided among those objects, so the sum of vessel capacities equals the configured total. After the simulation, a 15-minute forecast is calculated from the final aggregate state.

In [ ]:
terminal_config = {
    'num_cranes': 4,
    'num_vessels': 2,
    'num_yard_blocks': 12,
    'num_trucks': 48,
    'total_containers': 1000,
    'simulation_duration': 300,
    'sync_frequency': 1.0,
    'optimization_frequency': 30.0,
}

digital_twin = IntegratedDigitalTwin(terminal_config)

print("Digital twin initialized with:")
print(f"  Quay cranes: {terminal_config['num_cranes']}")
print(f"  Vessels: {terminal_config['num_vessels']}")
print(f"  Yard blocks represented in aggregate state: {terminal_config['num_yard_blocks']}")
print(f"  Trucks: {terminal_config['num_trucks']}")
print(f"  Total containers: {terminal_config['total_containers']}")
print(f"Subsystem state objects: {list(digital_twin.subsystems)}")

start_time = time.time()
sync_steps = int(terminal_config['simulation_duration'] / terminal_config['sync_frequency'])
optimization_interval_steps = max(
    1,
    int(terminal_config['optimization_frequency'] / terminal_config['sync_frequency']),
)

for step in range(sync_steps):
    simulated_time = start_time + step * terminal_config['sync_frequency']
    digital_twin.synchronize_subsystems(simulated_time)

    if step % optimization_interval_steps == 0:
        digital_twin.run_global_optimization(simulated_time)

prediction = digital_twin.predict_operations(horizon_minutes=15)

print(f"\nSynchronization steps recorded: {len(digital_twin.sync_history)}")
print(f"Global optimization calls recorded: {len(digital_twin.optimization_history)}")
print(f"Final simulated containers processed: {digital_twin.global_state['total_containers_processed']}")
print(f"Final active cranes: {digital_twin.global_state['active_cranes']}")
print(f"Final yard utilization: {digital_twin.global_state['yard_utilization']:.3f}")
print(f"Final truck dispatch efficiency: {digital_twin.global_state['truck_efficiency']:.3f}")
print(f"15-minute illustrative throughput forecast: {prediction['predicted_throughput']}")
print(f"Illustrative forecast interval: {prediction['confidence_interval']}")
print(f"Forecast risk factors: {prediction['risk_factors']}")

### Interpreting the simulation

The histories contain simulated observations, not measurements from a terminal. `sync_history` records one set of sampled synchronization metrics per simulated update. `optimization_history` records the bottlenecks, opportunities, recommendations, and rule-based effects associated with each optimization call. The final aggregate values summarize the state after the last update, while the forecast is calculated from that aggregate state and the selected horizon.

The `expected_improvement` entries are labels attached to recommendations and are converted into reported fractions when a rule is applied. They are not measured changes in throughput or efficiency. Likewise, the horizon factors used by `predict_operations` are fixed assumptions and cannot be interpreted as forecast-accuracy statistics. To establish empirical performance, the assumptions would need to be calibrated and evaluated against operational data; that evaluation is outside this simulation's scope.

### Relationship to dual-cycling quay crane operations

The crane state includes `dual_cycle_available`, and the opportunity check can recommend making dual-cycling opportunities available across the simulated cranes. This is an integration-level demonstration of how a quay-crane concept can be surfaced to a terminal-wide recommendation layer. The code does not construct individual container move pairs, calculate crane travel distances, or optimize a sequence of dual cycles. Therefore, its output should be read as a system-of-systems simulation with a rule-based coordination example, not as a detailed quay-crane scheduling solution.

### Modeling limitations

- Subsystem updates use simplified random transitions and do not enforce a complete physical or operational model.
- The crane position and interference-zone fields are stored but are not used to calculate crane travel or inter-crane interference.
- The recommendation rules are fixed threshold checks; they do not solve a constrained optimization problem.
- The simulated synchronization metrics are generated locally and do not establish communication latency, uptime, or data consistency in an operational deployment.
- The forecast uses illustrative coefficients and does not learn from the recorded histories.

These limitations define what can and cannot be concluded from the output: it can illustrate state integration and the order of the rule-based workflow, but it cannot substantiate real-world throughput gains, predictive accuracy, or reliability.

## Summary

This notebook demonstrates an integrated simulation with five represented terminal subsystem types: quay cranes, vessel schedules, aggregate yard management, truck dispatch, and resource allocation. The implementation follows a clear data flow:

1. Build typed subsystem state from the terminal configuration.
2. Advance subsystem states and aggregate selected measures at each simulated synchronization step.
3. Identify bottlenecks and opportunities using explicit rules.
4. Generate and apply the corresponding recommendations.
5. Calculate a horizon-based illustrative forecast from the final aggregate state.

The principal learning outcome is how subsystem state, synchronization, rule-based recommendations, and prediction can be organized in one system-of-systems framework. The simulation's numerical outputs depend on seeded random draws and stated assumptions. They are examples produced by the model, not validated operational results or evidence of measured performance.